In [1]:
## upload Mattsco dataset

import pandas as pd
from dotenv import load_dotenv
import os
load_dotenv()

df_final = pd.read_excel(r'data\Mattsco Product Parsed.Marty.12.19.24.xlsx')

print('len of df: ', len(df_final))

len of df:  28192


In [2]:
# UPLOAD VECTORIZER

ENCODER = os.getenv("ENCODER")

"""
https://huggingface.co/intfloat/multilingual-e5-base
https://huggingface.co/spaces/mteb/leaderboard
https://pytorch.org/get-started/locally/
"""

import joblib

with open(ENCODER, "rb") as f:
    encoder = joblib.load(f)

def get_encoded_embedding(text, encoder):
    if text is not None:
        return encoder(text).astype(float)
    
desc_to_vectorize = "0.5 90 DEGREE ELBOW , ASME B16.11 , CLASS 6000 , SOCKET WELDED END , ASTM A350 GRADE LF2 CLASS 1"

v = get_encoded_embedding(desc_to_vectorize, encoder=encoder)

#print('v is: ', v)
print('len of v: ', len(v))

c:\Users\endle\miniconda3\envs\qdrant-env\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


len of v:  768


In [3]:
# CONNECT TO QDRANT
import os
import json
import qdrant_client
from qdrant_client.http.models import Filter, FieldCondition, MatchValue,  MatchAny, PointStruct, VectorParams, Distance
from qdrant_client import QdrantClient


ENDLESSFORMS_QDRANT_URL = os.getenv("ENDLESSFORMS_QDRANT_URL")
ENDLESSFORMS_TEST_CLUSTER_KEY = os.getenv("ENDLESSFORMS_TEST_CLUSTER_KEY")

qdrantclient = QdrantClient(
    url=ENDLESSFORMS_QDRANT_URL,
    api_key=ENDLESSFORMS_TEST_CLUSTER_KEY,
)

print(qdrantclient)

collections = qdrantclient.get_collections()
list(collections)[0][1]

[CollectionDescription(name='TEXAS_PIPE_EXP'),
 CollectionDescription(name='test_intfloat_e5-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='test_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_HYBRID'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_BAAI_bge-reranker-v2-m3'),
 CollectionDescription(name='DENSE_VECTOR_BENCHMARK_25K_DEC17'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_TEST_DEC15_2'),
 CollectionDescription(name='ENCODER-TEST-INTFLOAT-MULTILINGUAL-E5-BASE'),
 CollectionDescription(name='ENCODER_TEST_2024-12-20_intfloat_e5-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-26_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_TEST_DEC15'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_sentence-transformers_all-mpnet-base-v2'),
 CollectionDescription(name='ENCODER

In [4]:
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import VectorParams

COLLECTION_NAME = "MATTSCO-MULTLINGUAL-E5-BASE"

# Create the collection
if not qdrantclient.collection_exists(COLLECTION_NAME):
    qdrantclient.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE, on_disk = True)
    )

collections = qdrantclient.get_collections()
list(collections)[0][1]

[CollectionDescription(name='TEXAS_PIPE_EXP'),
 CollectionDescription(name='test_intfloat_e5-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='test_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_HYBRID'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_BAAI_bge-reranker-v2-m3'),
 CollectionDescription(name='DENSE_VECTOR_BENCHMARK_25K_DEC17'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_TEST_DEC15_2'),
 CollectionDescription(name='ENCODER-TEST-INTFLOAT-MULTILINGUAL-E5-BASE'),
 CollectionDescription(name='ENCODER_TEST_2024-12-20_intfloat_e5-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-26_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_TEST_DEC15'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_sentence-transformers_all-mpnet-base-v2'),
 CollectionDescription(name='ENCODER

In [5]:
## clean data

from EnhancedDataCleanser import DataCleanser

cleanser = DataCleanser(
        default_uppercase=True,
        measurement_standardization=True,
        fraction_conversion=True
    )

df_clean = cleanser.clean_dataframe(df_final)

c:\Users\endle\Desktop\encoder-study\EnhancedDataCleanser.py:274: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df[col] = df[col].apply(lambda x: self.clean_text(x, **kwargs))


Removed 5533 duplicate rows


In [6]:
df_clean.columns

Index(['PRODUCT CODE', 'DESCRIPTION', 'CATEGORY', 'TYPE', 'PRIMARY SIZE',
       'REDUCING SIZE', 'LENGTH', 'MATERIAL NAME', 'MATERIAL SPECIFICATION',
       'MATERIAL GRADE', 'PRESSURE CL', 'PRIMARY SCHEDULE',
       'REDUCING SCHEDULE', 'MANUFACTURING PROCESS', 'END FINISH',
       'END CONNECTIONS', 'TRIM', 'MISCELLANEOUS', 'CONFIDENCE',
       'ORIGINAL_TEXT', 'PRIMARY WALL', 'REDUCING WALL', 'WALL THICKNESS',
       'MAX', 'OD', 'ID', 'WALL', 'MESSAGE', 'NOTE', 'ERROR', 'ERROR',
       'FOR EXAMPLE YOU MIGHT PROVIDE SOMETHING LIKE', 'FIRST_END',
       'SECOND_END', 'END1', 'END2',
       'FOR EXAMPLE A DESCRIPTION MIGHT LOOK LIKE',
       'TO PROCEED I WOULD NEED AN ACTUAL COMPONENT DESCRIPTION TO EXTRACT DATA ACCORDING TO THE PROVIDED SCHEMA THIS COULD BE SOMETHING LIKE',
       'FOR EXAMPLE YOU COULD PROVIDE A DESCRIPTION LIKE'],
      dtype='object')

In [8]:
print(len(df_clean))

22659


In [7]:
## now load sparse vectors into collection
from qdrant_client import QdrantClient, models

import numpy as np

# LOAD DATA
# this is for single pointstruct loading
# Initialize vectorizer


bad_data_points = []

"""input dataframe, output PointSturct"""
for i, row in df_clean.iterrows():
    payload = {
        "ID": str(i),
        "PRODUCT_CODE":str(row['PRODUCT CODE']),
        "DESCRIPTION":str(row['DESCRIPTION']),
        "CATEGORY": str(row['CATEGORY']),
        "TYPE": str(row['TYPE']),
        "PRIMARY_SIZE": str(row['PRIMARY SIZE']),
        "REDUCING_SIZE": str(row['REDUCING SIZE']),
        "LENGTH": row['LENGTH'],
        "MATERIAL_NAME": str(row['MATERIAL NAME']),
        "MATERIAL_SPECIFICATION": row['MATERIAL SPECIFICATION'],
        "MATERIAL_GRADE": row['MATERIAL GRADE'],
        "PRESSURE_CLASS": row['PRESSURE CL'],
        "PRIMARY_SCHEDULE": row['PRIMARY SCHEDULE'],
        "REDUCING_SCHEDULE": row['REDUCING SCHEDULE'],
        "MANUFACTURING_PROCESS": row['MANUFACTURING PROCESS'],
        "END_FINISH": row['END FINISH'],
        "END_CONNECTIONS": row['END CONNECTIONS'],
        "TRIM": row['TRIM'],
        "MISCELLANEOUS": row['MISCELLANEOUS'],
        "CONFIDENCE": row['CONFIDENCE'],
        "PRIMARY_WALL": row['PRIMARY WALL'],
        "REDUCING_WALL": row['REDUCING WALL'],
        "WALL_THICKNESS": row['WALL THICKNESS'],
        "OD": row['OD'],
        "ID": row['ID'],
    }

    desc = row['DESCRIPTION']

    if isinstance(desc,str):

        if desc != np.nan or desc != 'nan':

            v = get_encoded_embedding(desc,encoder=encoder)

            operation_info = qdrantclient.upsert(
                collection_name=COLLECTION_NAME,
                points=[
                    models.PointStruct(
                        id=i,
                        payload=payload,  # Add any additional payload if necessary
                        vector=v
                    )
                ],
            )

            print(operation_info)

    else:
        print('got a bad data point!')
        bad_data_points.append(payload)

print('all files uploaded!')
print('bad datapoints:', bad_data_points)

operation_id=0 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=1 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=2 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=3 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=4 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=5 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=6 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=7 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=8 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=9 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=10 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=11 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=12 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=13 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=14 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=15 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=16 status=<UpdateStat